# TB NN & TNN model -- Figures 1-4

MoS$_2$ (GGA), NN and TNN models, at $B = 0$ and with the Zeeman Hamiltonian at an enlarged field $\mu_B B = 80$ meV ($B \approx 1.4\times10^3$ T) so that the splitting is visible. Only the Zeeman Hamiltonian enters here, no moment shift.

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper setting, with a quick setting in the comment. Nothing here is cached; the whole notebook takes about 20 s.

In [ ]:
N_geo = 600        # square grid for the maps of Figs. 2-4; 200 for a quick look

## $\Gamma$-$K$-$M$-$\Gamma$ path

In [ ]:
def k_path(corners, n_total=600):
    corners = [np.asarray(c, dtype=float) for c in corners]
    seg = [np.linalg.norm(c2 - c1) for c1, c2 in zip(corners[:-1], corners[1:])]

    k_pts, x_pts, nodes = [], [], [0.0]
    for (c1, c2), s in zip(zip(corners[:-1], corners[1:]), seg):
        n = max(2, int(round(n_total * s / sum(seg))))
        f = np.linspace(0.0, 1.0, n, endpoint=False)
        k_pts.append(c1 + f[:, None] * (c2 - c1))
        x_pts.append(nodes[-1] + f * s)
        nodes.append(nodes[-1] + s)

    k_pts.append(corners[-1][None, :])
    x_pts.append(np.array([nodes[-1]]))
    return np.vstack(k_pts), np.concatenate(x_pts), np.array(nodes)


Gamma_pt = np.array([0.0, 0.0])
K_pt = np.array([K_len, 0.0])
M_pt = np.array([np.pi, np.pi / np.sqrt(3)])
k_line, x_line, x_nodes = k_path([Gamma_pt, K_pt, M_pt, Gamma_pt])

cases = [("NN", 0), ("TNN", 0), ("NN", B80), ("TNN", B80)]
tags = {0: r"$B=0$", B80: r"$\mu_BB=80$ meV"}

## Figure 1 -- band structure along $\Gamma$-$K$-$M$-$\Gamma$

Spin up blue, spin down red, energies measured from the valence-band maximum. The $B = 0$ bands are drawn in grey under the Zeeman panels.

In [ ]:
path_bands = {}
for model, B in cases:
    p = params("MoS2", "GGA", model)
    H = hamiltonian(model, p, k_line[:, 0], k_line[:, 1])
    up, dn = [np.linalg.eigvalsh(H_s) for H_s in zeeman(H, lambda_SOC["MoS2"], B)]
    VBM = max(up[:, 0].max(), dn[:, 0].max())
    path_bands[model, B] = (up - VBM, dn - VBM)

H_K = hamiltonian("TNN", params("MoS2"), np.array([K_len]), np.array([0.0]))
E_up, E_dn = [np.linalg.eigvalsh(H_s)[0] for H_s in zeeman(H_K, lambda_SOC["MoS2"], 0)]
print(f"In the TNN model the gap at K is {E_up[1] - E_up[0]:.4f} eV and the valence spin splitting is {E_up[0] - E_dn[0]:.4f} eV.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(TW, 2.25), sharey=True, gridspec_kw=dict(wspace=0.06))
for j, (model, B) in enumerate(cases):
    ax = axes[j]
    up, dn = path_bands[model, B]
    if B != 0:
        for E in path_bands[model, 0]:
            ax.plot(x_line, E, color="0.78", lw=0.8, zorder=1)
    for n in range(3):
        ax.plot(x_line, up[:, n], color=BLUE, lw=1.1, zorder=3, label=r"$\uparrow$" if n == 0 else None)
        ax.plot(x_line, dn[:, n], color=RED, lw=1.1, ls=(0, (4, 2)), zorder=2, label=r"$\downarrow$" if n == 0 else None)

    ax.set_xlim(x_line[0], x_line[-1])
    ax.set_ylim(-1.35, 4.05)
    ax.set_xticks(x_nodes)
    ax.set_xticklabels([r"$\Gamma$", "K", "M", r"$\Gamma$"])
    ax.minorticks_off()
    for x in x_nodes[1:-1]:
        ax.axvline(x, color="0.85", lw=0.5, zorder=0)
    ax.axhline(0, color="0.85", lw=0.5, zorder=0)
    ax.set_title(f"{model}, " + tags[B], pad=3)
    label(ax, "abcd"[j], x=0.04, y=0.96, va="top")

axes[0].set_ylabel("Energy (eV)")
axes[0].set_yticks(range(-1, 5))
axes[3].legend(loc="upper right", handlelength=1.6)
save(fig, "bands_grid")
plt.show()

## Berry curvature and quantum metric over the zone

The occupied bands are the valence bands of the two blocks, so $\Omega = \Omega_{v\uparrow} + \Omega_{v\downarrow}$ and
$$g^{ab}(\mathbf{k}) = \sum_{v\,\in\,{\rm occ}}\sum_c \frac{\mathrm{Re}[v^a_{vc}v^b_{cv}]}{(E_c - E_v)^2}.$$
About 20 s at paper settings.

In [ ]:
k_x, k_y, BZ = bz_grid(N_geo)[0]
dk = k_x[1] - k_x[0]
k_xv, k_yv = np.meshgrid(k_x, k_y)

geometry, chern = {}, {}
for model, B in cases:
    p = params("MoS2", "GGA", model)
    Omega_occ, g_occ = 0, 0
    for s, H in zip(("up", "dn"), zeeman(hamiltonian(model, p, k_xv, k_yv), lambda_SOC["MoS2"], B)):
        E, V, v_kx, v_ky = bands(H, dk)
        Omega, M = berry_and_moment(E, v_kx, v_ky, p["a"])
        Omega_occ = Omega_occ + Omega[..., 0]
        g_occ = g_occ + metric(E, v_kx, v_ky, 1) + metric(E, v_kx, v_ky, 2)
        if B == 0:
            chern[model, s] = [np.sum(Omega[..., n][BZ]) * dk**2 / (2 * np.pi) for n in range(3)]
    tr_g = g_occ[..., 0, 0] + g_occ[..., 1, 1]
    det_g = g_occ[..., 0, 0] * g_occ[..., 1, 1] - g_occ[..., 0, 1]**2
    geometry[model, B] = (Omega_occ, tr_g, det_g)

## Chern numbers

In [ ]:
for model in ("NN", "TNN"):
    for s in ("up", "dn"):
        c = chern[model, s]
        print(f"The Chern numbers of the {model} model (spin {s}) are {fmt(c[0])}, {fmt(c[1])}, and {fmt(c[2])}.")

## Figures 2-4 -- occupied Berry curvature, metric trace and metric determinant

Same four cases as Fig. 1, over the window $|k_{x,y}| < 1.16\cdot4\pi/3$ with the zone boundary drawn.

In [ ]:
L = 1.16 * K_len
maps = [("berry_grid_1", 0, "bwr", -3.5, 3.5, r"$\Omega(\mathbf{k})$ ($a^2$)"),
        ("metric_trace_grid", 1, "viridis", 0, 4.0, r"$\mathrm{tr}\,g(\mathbf{k})$ ($a^2$)"),
        ("metric_det_grid", 2, "viridis", 0, 4.0, r"$\det g(\mathbf{k})$ ($a^4$)")]

for name, idx, cmap, vmin, vmax, cb_label in maps:
    fig = plt.figure(figsize=(TW, 1.95))
    gs = gridspec.GridSpec(1, 5, width_ratios=[1, 1, 1, 1, 0.05], wspace=0.06, left=0.0, right=0.93, top=0.86, bottom=0.02)
    for j, (model, B) in enumerate(cases):
        ax = fig.add_subplot(gs[0, j])
        Z = geometry[model, B][idx]
        im = ax.pcolormesh(k_xv, k_yv, Z, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", rasterized=True)
        ax.contour(k_xv, k_yv, Z, levels=np.linspace(vmin, vmax, 9), colors="w" if cmap == "viridis" else "0.35",
                   linewidths=0.3, linestyles="dashed", alpha=0.6)
        ax.plot(hex_x, hex_y, "k-", lw=0.9)
        ax.set_aspect("equal")
        ax.set_xlim(-L, L)
        ax.set_ylim(-L, L)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.minorticks_off()
        ax.set_title(f"{model}, " + tags[B], pad=3, loc="right")
        label(ax, "abcd"[j], y=1.03)
    cb = fig.colorbar(im, cax=fig.add_subplot(gs[0, 4]))
    cb.set_label(cb_label)
    cb.outline.set_linewidth(0.5)
    save(fig, name)
    plt.show()